# LangChain 기초

**학습 목표**

> 1. 이번 실습에서는 LLM 앱을 만들 때 가장 먼저 필요한 LangChain 기본 구성요소를 익힌다.
> 2. `init_chat_model`을 사용해 모델 식별자만 변경하여 OpenAI, Gemini, Ollama 간 전환을 수행한다.
> 3. **Messages → Prompt Template → Output Parser** 의 표준 처리 흐름을 익힌다.
> 4. **LLM을 호출하고, 프롬프트를 템플릿화하고, 결과를 원하는 형식으로 받는 법**을 익힌다.


---

> **LangChain v1.0의 변경점**
> - `langchain` 네임스페이스가 5개로 단순화됨: `langchain.messages`, `langchain.tools`, `langchain.agents`, `langchain.chat_models`, `langchain.embeddings`.
> - `LLMChain`, 전통 Retriever 등 레거시 기능은 `langchain-classic` 패키지로 분리됨.
> - 본 실습은 v1.0 이상의 API만 사용합니다.
---

# 1. 환경 준비


## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.

In [1]:
# 필요한 라이브러리 설치
%pip install -U langchain langchain-core langchain-openai langchain-google-genai langchain-groq langchain-ollama python-dotenv pydantic pandas

  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached xxhash-3.7.0-cp313-cp313-win_amd64.whl.metadata (13 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.5.9-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached google_genai-2.8.0-py3-none-any.whl.metadata (53 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyasn1-0.6.3-py3-none-any.whl.metadata (8.4 kB)
   ---------------------------------------- 0.0/550.1 kB ? eta -:--:--
   ---------------------------------------- 550.1/550.1 kB 9.8 MB/s  0:00:00
Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl (154 kB)
   ---------------------------------------- 0.0/874.8 kB ? eta -:--:--
   ---------------------------------------- 874.8/874.8 kB 9.6 MB/s  0:00:00
Using cache

## (2) 라이브러리 Import

이번 실습에서 사용하는 핵심 객체는 다음과 같습니다.

| 객체 | 역할 |
|---|---|
| `init_chat_model` | `provider:model` 문자열로 여러 제공자의 채팅 모델을 통합 초기화 |
| `PromptTemplate` | 문자열 기반 프롬프트 템플릿 |
| `ChatPromptTemplate` | system/human 메시지를 분리하는 채팅 프롬프트 |
| `StrOutputParser` | 모델 응답에서 문자열만 추출 |


In [2]:
import os
from pathlib import Path
from getpass import getpass
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## (3) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [6]:
load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_API_KEY: 있음
GOOGLE_API_KEY: 없음


# 2. LangChain이 필요한 이유

OpenAI SDK나 다른 LLM API만으로도 모델 호출은 가능합니다. 예를 들어 "랭체인이 뭐야?"라고 바로 물어볼 수 있습니다.

하지만 실제 서비스나 업무 자동화에서는 단순 호출만으로 충분하지 않습니다.

| 필요 기능 | 실제 상황 예시 |
|---|---|
| 프롬프트 재사용 | 주제만 바꿔 같은 형식의 설명 생성 |
| 역할 부여 | 강사, 면접관, 상담사, 분석가 역할 지정 |
| 출력 형식 고정 | JSON, 표, 리스트, Pydantic 객체 |
| 여러 단계 연결 | 요약 -> 번역 -> 퀴즈 생성 |
| 대량 처리 | 고객 후기 100개를 같은 방식으로 분석 |
| 유지보수 | 프롬프트, 모델, 파서를 분리해 관리 |

LangChain은 LLM 호출을 **구조화된 실행 블록**으로 만들고, 이 블록들을 연결해 앱의 흐름을 구성하는 도구입니다.

# 3. Model

## (1) Model과 `init_chat_model`

- Model은 실제 답변을 생성하는 엔진
- LangChain에서는 모델을 공통 인터페이스로 감싸기 때문에, 이후 프롬프트나 파서와 쉽게 연결할 수 있음

- LangChain v1.x에서는 `init_chat_model("provider:model")` 한 함수로 여러 제공자의 채팅 모델을 통합 초기화할 수 있음.
    - 제공자별 import를 줄일 수 있습니다.
    - 모델 문자열만 바꾸면 OpenAI, Gemini, Groq, Ollama 등으로 전환할 수 있습니다.
    - 이후 `invoke`, `stream`, `batch`, Prompt 연결 방식은 동일하게 유지됩니다.


`CHAT_MODEL`은 `provider:model` 형식을 권장합니다.

| 제공자 | 예시 |
|---|---|
| OpenAI | `openai:gpt-4.1-mini` |
| Gemini | `google_genai:gemini-2.5-flash-lite` |
| Ollama | `ollama:gemma4:e4b` |

### [참고] GPT-5 시리즈는 temperature 고정

GPT-5 / 5.4 / 5.5 모델은 내부에서 multi-pass reasoning 을 수행하기 때문에 답의 무작위성을 막는 `temperature`·`top_p`·`logprob` 파라미터가 사실상 사용 불가 (temperature 는 1 로 고정). `init_chat_model("openai:gpt-5.4-mini")` 처럼 추가 인자 없이 부르면 됩니다.

답의 다양성·톤 조정은 **system 프롬프트**와 **few-shot 예시**로 합니다.

- `init_chat_model`로 만든 모델도 다른 채팅 모델과 동일하게 `invoke()`로 호출합니다. 반환값은 메시지 객체이며, 실제 답변 텍스트는 `.content`에서 확인합니다.

## (2) `ChatOpenAI` 사용

- `invoke()`는 하나의 입력을 넣고 하나의 출력을 받는 가장 기본적인 실행 방식입니다.

- 응답 객체에는 답변 본문뿐 아니라 모델명, 토큰 사용량 같은 메타데이터가 포함될 수 있습니다. 
- 실제 서비스에서는 사용량 추적이나 로깅에 활용할 수 있습니다.

In [ ]:
print("content:", )
print("response_metadata:", )
print("usage_metadata:", )

## (3) Temperature 비교

- `temperature`는 답변의 다양성과 예측 가능성을 조절하는 값

| 값 | 특징 |
|---|---|
| 0에 가까움 | 안정적, 반복 실행 시 비슷한 결과 |
| 1에 가까움 | 다양하고 창의적인 결과 |


In [ ]:

print("[temperature=0]")
print()

print("\n[temperature=1]")
print()

## [실습] 모델 호출 바꿔보기

모델을 gemini 등 다른 모델로 바꿔봅니다.
그리고 아래 질문을 바꿔 실행해 봅니다.

- "RAG를 비전공자에게 설명해줘."
- "AI Agent를 회사 업무 자동화 예시로 설명해줘."
- "LangGraph와 LangChain의 차이를 간단히 설명해줘."

# 4. Prompt

- Prompt는 LLM에게 주는 작업 지시서
- 좋은 프롬프트는 단순히 질문을 잘 쓰는 것이 아니라, 다음 요소를 설계하는 일

| 요소 | 예시 |
|---|---|
| 역할 | "너는 AI 강의를 하는 친절한 강사야." |
| 목적 | "초보자가 이해할 수 있게 설명해줘." |
| 제약 | "5문장 이내로 작성해줘." |
| 출력 형식 | "표로 정리해줘.", "JSON으로 답해줘." |
| 예시 | "아래 예시와 같은 톤으로 작성해줘." |

LangChain에서는 반복되는 프롬프트를 템플릿으로 만들고, 변수만 바꿔 재사용합니다.

## (1) PromptTemplate

- `PromptTemplate`은 문자열 기반 템플릿
- `{topic}` 같은 변수를 넣고, 실행할 때 실제 값을 전달

## (2) ChatPromptTemplate

- 채팅 모델에는 `ChatPromptTemplate`이 더 자주 사용됨
- 시스템 메시지, 사용자 메시지, AI 메시지 등 역할(role) 구분
- 다중 메시지 기반의 프롬프트 흐름을 구성할 수 있도록 도와주는 템플릿

| 메시지 역할 | 의미 |
|---|---|
| `System` | 모델의 역할, 원칙, 톤. AI에게 역할/성격을 지정 |
| `Human` | 실제 사용자 질문 또는 요청 |
| `AI` | AI 응답 |

## [실습] 역할 프롬프트 만들기

`system` 역할을 바꿔 같은 주제의 답변이 어떻게 달라지는지 확인합니다.

예시 역할:

- "너는 초등학생에게 설명하는 과학 선생님이다."
- "너는 기업 임원에게 보고하는 AI 컨설턴트다."
- "너는 개발자에게 코드 중심으로 설명하는 시니어 엔지니어다."

## [실습] 영화 추천 템플릿 만들기
- 입력변수 : 장르
- 장르를 입력받아 영화 1편과 추천이유를 설명하는 템플릿을 만들고 사용해 봅시다.

# 5. Output Parser

- Output Parser는 LLM에서 반환된 자유형 텍스트(string)를 우리가 원하는 형태로 가공해주는 도구

- LLM의 응답은 기본적으로 메시지 객체이고, 사람이 읽을 때는 `response.content`만 확인하면 되지만, 프로그램에서는 결과를 일정한 형태로 다루는 것이 중요함


| Parser | 역할 | 사용 상황 |
|---|---|---|
| `StrOutputParser` | 응답에서 문자열만 추출 | 일반 답변, 이메일, 요약 |
| JSON 계열 Parser | JSON 문자열을 dict로 변환 | API 응답처럼 쓰고 싶을 때 |
| Pydantic 구조화 출력 | 스키마에 맞는 객체로 검증 | 분류, 추출, 업무 자동화 |

최근 LangChain에서는 모델의 `with_structured_output()` 기능을 사용해 Pydantic 모델로 결과를 받는 방식도 많이 사용합니다.

## (1) StrOutputParser

`StrOutputParser`를 체인 끝에 붙이면 모델 응답 객체에서 텍스트만 꺼내 줍니다.

## (2) PydanticOutputParser

#### 1) Pydantic
- 파이썬에서 데이터 형태를 정의하고 검증하는 라이브러리

In [ ]:
class User(BaseModel):
    name: str
    age: int

#### 2) 출력파서로 이용
- llm의 성능에 따라 출력 파싱에 맞게 적절한  답변을 할 수도 있고, 잘못 답변해서 오류가 날 수도 있음.

## (3) 구조화 출력

업무 자동화에서는 자유 텍스트보다 정해진 구조가 더 유용할 때가 많습니다.

예를 들어 고객 후기를 분석한다면 다음처럼 감성, 카테고리, 우선순위, 다음 조치를 분리해서 받아야 이후 시스템에서 활용하기 쉽습니다.

In [ ]:
class CustomerIssue(BaseModel):
    sentiment: 

    category: 
    priority: 
    summary: 
    next_action: 


issue_model =

review = 
issue = 

issue

## [실습] 감성 분석 결과 구조화

아래 리뷰를 분석해 다음 필드를 가진 `MovieReview` 모델을 만들어 봅니다.

입력 문장:

```text
이 영화는 영상미는 좋았지만 스토리가 너무 지루했다.
```

출력 목표:

```json
{
  "sentiment": "mixed",
  "positive": "영상미가 좋음",
  "negative": "스토리가 지루함",
  "recommendation": "시각적 연출을 좋아하는 관객에게만 추천"
}
```

## [실습] 게임 캐릭터 카드
- 게임 캐릭터 이름, 직업, 성격, 대표 특기 또는 필살기, 약점를 나타내는 게임 캐릭터 카드를 만들어보세요.